# Title

Angos and Tan

BSDSBA 2028

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

### Data Download

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("usdot/flight-delays", output_dir="./data")

# print("Path to dataset files:", path)

# OVERVIEW

This mini-project analyzes airline operations using the **U.S. Department of Transportation (DOT) 2015 Flight Delays and Cancellations dataset**. The project investigates how airport congestion, airline scheduling practices, and traffic density contribute to flight delays. In addition to identifying structural causes of delays, the project aims to forecast flight delay duration based on operational conditions, including airport traffic, route characteristics, and schedule realism.

Airline delays are often attributed to weather or operational disruptions, but underlying causes may include unrealistic scheduling and airport capacity constraints. By analyzing patterns in flight schedules, traffic density, and delay distributions, this project aims to identify systemic patterns that contribute to delays and congestion.

Understanding these factors has practical applications for airlines, airports, and passengers. Airlines can improve schedule planning, airports can identify congestion thresholds, and travelers can better understand which routes or airports are more reliable.


## GOALS

1. Analyze patterns in flight delays across airlines, routes, airports, and time of day.

2. Identify congestion thresholds at which airport traffic density results in significant increases in delay.

3. Forecast expected delay duration for flights using operational features such as airport traffic, route reliability, and schedule realism.

4. Evaluate whether airline schedules are realistic by comparing scheduled flight time with actual flight durations.


## Basic

### Data Loading

In [ ]:
# Load Data
airlines = pd.read_csv("data/airlines.csv")
airports = pd.read_csv("data/airports.csv")
flights = pd.read_csv("data/flights.csv", low_memory=False)

In [ ]:
display(flights.head())
print(flights.info())
print(flights.shape)

### DATA CLEANING

#### Fix 5-digit airport codes to standard 3-character IATA codes

Some data points use 5-digit airport codes instead of the standard IATA codes:

In [ ]:
flights[flights['ORIGIN_AIRPORT'].str.len()!=3].head()[['ORIGIN_AIRPORT','DESTINATION_AIRPORT']]

Also, comparing airports present in our `flights` and `airports` dataframe, one airport is missing. To solve this, that airport will be removed from the list of flights entirely. 

We'll utilize a helper function to fix these two issues.

In [ ]:
# import clean_flights

# clean_flights.clean('data/flights.csv')

### FEATURE ENGINEERING

The cleaned flights will be used instead of the original csv file.

In [ ]:
flights = pd.read_csv("data/flights_filtered_fixed.csv", low_memory=False)

### Airport-level features:

- `avg_flights_per_hour`: $\frac{\text{total flights for airport}}{\text{number of hours observed}}$
- `avg_delay_rate`: P(delay)

In [ ]:
filtered_flights = flights[
    (flights['CANCELLED'] == 0) &
    (flights['DIVERTED'] == 0) &
]


airports['avg_flights_per_hour'] = airports['IATA_CODE'].map(
    flights.assign(hour=filtered_flights['SCHEDULED_DEPARTURE']//100)
           .groupby(['ORIGIN_AIRPORT','hour'])
           .size()
           .groupby('ORIGIN_AIRPORT')
           .mean()
           .round()
)

airports['avg_flights_per_hour'] = airports['IATA_CODE'].map(
    flights.assign(hour=filtered_flights['SCHEDULED_DEPARTURE']//100)
           .groupby(['ORIGIN_AIRPORT','hour'])
           .size()
           .groupby('ORIGIN_AIRPORT')
           .mean()
           .round()
)

airport_delay = (
    valid_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby('ORIGIN_AIRPORT')['delayed']
    .mean()
)

airports['AVG_DELAY_RATE'] = airports['IATA_CODE'].map(airport_delay)

display(airports.head())


### Route-level features

- `schedule_padding` - $\text{average scheduled time} - \text{median flight time}$

- `ave_delay` - $\frac{\sum \text{arrival delays}}{\text{num flights for the route}}$

- `delay-rate` - P(delay)

In [ ]:
route_df = (
    filtered_flights
    .groupby(['ORIGIN_AIRPORT','DESTINATION_AIRPORT'], as_index=False)
    .agg(
        NUM_FLIGHTS=('ELAPSED_TIME','size'),
        AVE_FLIGHT_TIME=('ELAPSED_TIME','mean'),
        MEDIAN_FLIGHT_TIME=('ELAPSED_TIME','median'),
        AVE_SCHEDULED_TIME=('SCHEDULED_TIME','mean')
    )
)

route_df['SCHEDULE_PADDING'] = (
    route_df['AVE_SCHEDULED_TIME'] -
    route_df['MEDIAN_FLIGHT_TIME']
)

route_df['AVE_DELAY'] = (
    filtered_flights
    .groupby(['ORIGIN_AIRPORT','DESTINATION_AIRPORT'])['ARRIVAL_DELAY']
    .mean()
    .values
)

delay_prob = (
    filtered_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby(['ORIGIN_AIRPORT','DESTINATION_AIRPORT'])['delayed']
    .mean()
)

route_df['DELAY_RATE'] = delay_prob.values

display(route_df.head())

## Airport reliability:

In [ ]:
display(airports.sort_values(by='AVG_DELAY_RATE', ascending=False).head(10).reset_index(drop=True)[['AIRPORT','AVG_DELAY_RATE']])

In [ ]:
px.scatter(
    airports,
    x = 'avg_flights_per_hour',
    y = 'AVG_DELAY_RATE',
    labels = {'avg_flights_per_hour': 'Average Flights per Hour',
              'AVG_DELAY_RATE': 'Average Delay Rate'},
    title = 'Airport-level Traffic to delay rate relationship',
    subtitle= 'There is a slight positive trend showing that delay rates increase as airports faces higher hourly flights.',
    trendline='ols'
)

In [ ]:
airport_hour_df = (
    valid_flights
    .assign(
        hour = valid_flights['SCHEDULED_DEPARTURE'] // 100,
        delayed = valid_flights['ARRIVAL_DELAY'] > 0
    )
    .groupby(['ORIGIN_AIRPORT','hour'])
    .agg(
        flights=('FLIGHT_NUMBER','size'),
        avg_delay=('ARRIVAL_DELAY', lambda x: x[x > 0].mean()),
        delay_rate=('delayed','mean')
    )
    .reset_index()
)

In [ ]:
airport_hour_df.sort_values(by='delay_rate')

In [ ]:
fig = px.scatter(
    airport_hour_df,
    x='hour',
    y='delay_rate',
    trendline='ols',
    opacity=0.4,
    hover_data = {'ORIGIN_AIRPORT': True}
)

fig.show()

### Visualizing Flights

The following code renders a graphic of flights by hour

In [ ]:
# Graphing every airport as points on the map
node_trace = go.Scattermap(
    lat=airports['LATITUDE'],
    lon=airports['LONGITUDE'],
    mode='markers',
    marker=dict(size=5, color='white', opacity=0.8),
    text=airports['IATA_CODE'] + ' — ' + airports['CITY'],
    hoverinfo='text',
    showlegend=False,
)

# Delay color/width bins 
DELAY_BINS   = [-np.inf, 0, 5, 10, 20, np.inf]
DELAY_COLORS = [
    'rgba(100,149,237,0.5)',  # blue      — on time / early
    'rgba(255,220,50,0.6)',   # yellow    — 1–15 min
    'rgba(255,140,0,0.7)',    # orange    — 15–30 min
    'rgba(220,40,40,0.8)',    # red       — 30–60 min
    'rgba(139,0,0,0.9)',      # dark red  — 60+ min
]
DELAY_WIDTHS = [0.5, 1, 2, 3, 4.5]

In [ ]:
def build_edge_traces(hour_flights):
    # Always initialize all 6 traces as empty
    on_time_lats, on_time_lons = [], []
    bin_lats  = {1: [], 2: [], 3: [], 4: [], 6: []}
    bin_lons  = {1: [], 2: [], 3: [], 4: [], 6: []}

    if not hour_flights.empty:
        routes = (
            hour_flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .agg(COUNT=('FLIGHT_NUMBER', 'count'), AVG_DELAY=('ARRIVAL_DELAY', 'mean'))
            .reset_index()
        )

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='ORIGIN_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'orig_lat', 'LONGITUDE': 'orig_lon'}).drop(columns='IATA_CODE')

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='DESTINATION_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'dest_lat', 'LONGITUDE': 'dest_lon'}).drop(columns='IATA_CODE')

        routes['DELAYED_COUNT'] = (
            hour_flights[hour_flights['ARRIVAL_DELAY'] > 5]
            .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .size()
            .reindex(pd.MultiIndex.from_frame(routes[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']]))
            .fillna(0)
            .values
        )

        on_time = routes[routes['DELAYED_COUNT'] == 0]
        delayed = routes[routes['DELAYED_COUNT'] > 0].copy()

        for _, row in on_time.iterrows():
            on_time_lats += [row['orig_lat'], row['dest_lat'], None]
            on_time_lons += [row['orig_lon'], row['dest_lon'], None]

        if not delayed.empty:
            delayed['width'] = pd.cut(
                delayed['DELAYED_COUNT'], bins=5,
                labels=[1, 2, 3, 4, 6],
                duplicates='drop'
            ).astype(float).fillna(1)

            for _, row in delayed.iterrows():
                w = row['width']
                bin_lats[w] += [row['orig_lat'], row['dest_lat'], None]
                bin_lons[w] += [row['orig_lon'], row['dest_lon'], None]


    traces = [
        go.Scattermap(lat=on_time_lats, lon=on_time_lons, mode='lines',
                      line=dict(width=0.5, color='rgba(100,149,237,0.2)'),
                      hoverinfo='none', showlegend=False),
    ]
    
    for w, color in zip([1, 2, 3, 4, 6], [
        'rgba(255,220,50,0.6)',
        'rgba(255,160,0,0.7)',
        'rgba(220,80,40,0.8)',
        'rgba(200,20,20,0.85)',
        'rgba(139,0,0,0.9)',
    ]):
        traces.append(go.Scattermap(
            lat=bin_lats[w], lon=bin_lons[w], mode='lines',
            line=dict(width=w, color=color),
            hoverinfo='none', showlegend=False,
        ))

    return traces  

In [ ]:
def generate_frames(flights):
    frames = []
    slider_steps = []

    min_day = int(flights['DAY'].min())
    max_day = int(flights['DAY'].max())

    for day in range(min_day, max_day + 1):
        for hour in range(24):
            label = f"Jan {day:02d}  {hour:02d}:00"
            slice_ = flights[(flights['DAY'] == day) & (flights['DEPARTURE_HOUR'] == hour)]
            edge_traces = build_edge_traces(slice_)

            frames.append(go.Frame(
                data=edge_traces + [node_trace],
                name=label
            ))
            slider_steps.append(dict(
                method='animate',
                args=[[label], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
                label=f"D{day} {hour:02d}h"
            ))

    return frames, slider_steps

In [ ]:
def build_figure(flights):
    frames, slider_steps = generate_frames(flights)

    min_day  = int(flights['DAY'].min())
    min_hour = int(flights['DEPARTURE_HOUR'].min())

    init_traces = build_edge_traces(
        flights[(flights['DAY'] == min_day) & (flights['DEPARTURE_HOUR'] == min_hour)]
    )

    fig = go.Figure(
        data=init_traces + [node_trace],
        frames=frames
    )

    _ = fig.update_layout(
        map=dict(style='carto-darkmatter', center=dict(lat=39, lon=-98), zoom=3),
        margin=dict(r=0, t=50, l=0, b=0),
        title=dict(text='US Flight Delays — Jan 1–7 2015 (Hourly)', font=dict(color='white')),
        paper_bgcolor='#1a1a2e',
        updatemenus=[dict(
            type='buttons', showactive=False, y=1.08, x=0.0,
            buttons=[
                dict(label='▶ Play', method='animate',
                     args=[None, dict(frame=dict(duration=400, redraw=True), fromcurrent=True)]),
                dict(label='⏸ Pause', method='animate',
                     args=[[None], dict(mode='immediate')]),
            ]
        )],
        sliders=[dict(
            steps=slider_steps,
            currentvalue=dict(prefix='Time: ', font=dict(size=13, color='white')),
            font=dict(color='white'),
            len=0.95, x=0.02,
        )]
    )

    return fig

## Intermediate

## Advanced